Versions of `pyiron_workflow` older than `0.17.0` pre-date the availability of `flowrep` and follow a slightly different paradigm. Old `@as_function_node`-decorated functions simply need to be re-decorated to `@fr.atomic` to continue operating, but `@as_macro_node`-decorated functions were parsed under different assumptions and cannot be directly replaced with `@fr.workflow` decorators.

To help you migrate from older versions of `pyiron_workflow`, we offer a compatibility module which lets you keep decorators of the same name, but transform the function into a creator for the _new_ node types.

Our hope is that by switching your import location for the `@as_function_node` and `@as_macro_node` tools to this compatibility layer, you can get a smooth

In [1]:
import pyiron_workflow as pwf

In [2]:
@pwf.compatibility.as_function_node("z")
def add(x, y):
    z = x + y
    return z

@pwf.compatibility.as_function_node("y")
def double(x):
    d = 2 * x
    return d

@pwf.compatibility.as_macro_node("summed")
def macro_inner(self, x, y):
    """Child built with keyword args."""
    self.s = add(x=x, y=y)
    return self.s

@pwf.compatibility.as_macro_node("total")
def macro_outer(self, x, y):
    """Macro child built with kwargs; the single-output macro is passed as input."""
    self.inner = macro_inner(x=x, y=y)
    self.combined = double(self.inner)
    return self.combined

Although they look like old-style definitions, these are now creators for new-style nodes

In [3]:
node = add()
print(type(node))
print(node.recipe.model_dump_json(indent=2))

<class 'pyiron_workflow.atomic_node.Atomic'>
{
  "type": "atomic",
  "inputs": [
    "x",
    "y"
  ],
  "outputs": [
    "z"
  ],
  "description": null,
  "reference": {
    "info": {
      "module": "__main__",
      "qualname": "add.decorated",
      "version": null
    },
    "inputs_with_defaults": [],
    "restricted_input_kinds": {}
  }
}


We can use `flowrep` to compile the resulting workflow recipes back into python code that can be used in place of your `@as_macro_node`-decorated functions with only some light massaging (updating imports, and replacing the upstream `@as_function_node` with `@fr.atomic`)

In [4]:
import flowrep as fr

In [5]:
macro = macro_outer()

rendered = fr.tools.flowrep2python(
    macro.recipe.model_copy(
        update={"reference": None}
    )
)
print(rendered.source)

from __future__ import annotations

import __main__
import flowrep
import typing

@flowrep.workflow("total")
def compiled_from_workflow_recipe(x, y):
    macro_inner = __main__.macro_inner.decorated(x=x, y=y)
    double = __main__.double.decorated(x=macro_inner)
    return double




This transformation process works for many, but not all `@as_macro_node`-decorated functions. E.g., macros using flow-controlls like for-loops will simply need to be re-written. Nonetheless, the formats are still quite similar, and we hope the transition is not burdensome. Due to the high similarity of the old and new formats, you may get good results providing an LLM a couple of concrete examples of the transformation, and asking it to follow similar patterns for the remaining nodes.

If you require any assistance migrating your nodes, please raise an issue [on GitHub](https://github.com/pyiron/pyiron_workflow/issues).